# Sao chép File, Folder Google Drive Siêu Nhanh (A to B)
Công cụ này giúp bạn copy toàn bộ thư mục được chia sẻ về Drive cá nhân trực tiếp trên máy chủ của Google, không tốn băng thông mạng của bạn và tốc độ cực kỳ nhanh (Siêu tốc).

### Bước 1: Chạy ô dưới đây để Cấp quyền truy cập Google Drive

In [ ]:
#@title Khởi tạo & Xác thực tài khoản
from google.colab import auth
try:
    auth.authenticate_user()
    from googleapiclient.discovery import build
    drive_service = build('drive', 'v3')
    print('✅ Xác thực thành công! Hãy tiếp tục chạy ô bên dưới.')
except Exception as e:
    print('❌ Lỗi xác thực:', e)

### Bước 2: Nhập Link và Bắt đầu Copy

In [ ]:
#@title Input - Đầu Vào
import re
import time
import ipywidgets as widgets
from IPython.display import display, clear_output

# Hàm trích xuất ID từ link Google Drive
def get_id_from_url(url):
    match = re.search(r'/folders/([a-zA-Z0-9_-]+)', url)
    if match: return match.group(1)
    match = re.search(r'id=([a-zA-Z0-9_-]+)', url)
    if match: return match.group(1)
    match = re.search(r'/d/([a-zA-Z0-9_-]+)', url)
    if match: return match.group(1)
    return url.strip() # Trường hợp user nhập thẳng ID

# Tạo giao diện nhập liệu giống trong video
dest_text = widgets.Text(description='Drive của bạn:', placeholder='Dán link thư mục đích trên Drive của bạn vào đây', layout=widgets.Layout(width='80%'))
source_text = widgets.Text(description='Drive chia sẻ:', placeholder='Dán link thư mục chia sẻ cần copy vào đây', layout=widgets.Layout(width='80%'))
run_btn = widgets.Button(description='▶ Khởi chạy Copy', button_style='info', icon='play')
out = widgets.Output()

display(dest_text)
display(source_text)
display(run_btn)
display(out)

def copy_folder(service, source_id, dest_id, path=""):
    total_files_copied = 0
    # Lấy danh sách file/folder trong thư mục nguồn
    results = service.files().list(
        q=f"'{source_id}' in parents and trashed=false",
        pageSize=1000, fields="nextPageToken, files(id, name, mimeType, size)").execute()
    items = results.get('files', [])
    
    if not items:
        return 0
    
    for item in items:
        if item['mimeType'] == 'application/vnd.google-apps.folder':
            print(f"📁 Tạo Folder: {path}/{item['name']} ...")
            file_metadata = {'name': item['name'], 'mimeType': 'application/vnd.google-apps.folder', 'parents': [dest_id]}
            new_folder = service.files().create(body=file_metadata, fields='id').execute()
            total_files_copied += copy_folder(service, item['id'], new_folder.get('id'), path + "/" + item['name'])
        else:
            size_mb = round(int(item.get('size', 0)) / (1024 * 1024), 2) if 'size' in item else 0
            print(f"📄 Copying: {item['name']} (Size: {size_mb} MB) ...")
            body = {'parents': [dest_id]}
            try:
                service.files().copy(fileId=item['id'], body=body).execute()
                total_files_copied += 1
            except Exception as e:
                print(f"⚠️ Lỗi khi copy {item['name']}: {e}")
    return total_files_copied

def on_run_clicked(b):
    with out:
        clear_output()
        source_id = get_id_from_url(source_text.value)
        dest_id = get_id_from_url(dest_text.value)
        if not source_id or not dest_id or source_id == "" or dest_id == "":
            print('❌ Vui lòng nhập đầy đủ link Drive của bạn và Drive chia sẻ!')
            return
        
        print('🚀 Bắt đầu quá trình copy siêu tốc trên server Google...')
        print('-' * 50)
        start_time = time.time()
        
        try:
            total = copy_folder(drive_service, source_id, dest_id)
            elapsed_time = round(time.time() - start_time, 2)
            print('-' * 50)
            print(f'✅ Done. Đã copy thành công {total} files.')
            print(f'⏱️ Total Time: {elapsed_time} s.')
        except Exception as e:
            print(f'\n❌ Đã xảy ra lỗi (có thể do sai link hoặc quyền truy cập): {e}')

run_btn.on_click(on_run_clicked)
